In [0]:
%sql
CREATE CATALOG IF NOT EXISTS housing_dev
-- If your workspace requires it, add your own storage path:
-- MANAGED LOCATION 'abfss://<container>@<storage-account>.dfs.core.windows.net/<path>'
;


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS housing_dev.landing;
CREATE SCHEMA IF NOT EXISTS housing_dev.bronze;
CREATE SCHEMA IF NOT EXISTS housing_dev.silver;
CREATE SCHEMA IF NOT EXISTS housing_dev.gold;
CREATE VOLUME IF NOT EXISTS housing_dev.landing.raw;

In [0]:
%sql
create table if not exists housing_dev.landing.zori as 
select * from read_files(
    '/Volumes/housing_dev/landing/raw/zori/',
  format => 'csv',
  header => true,
  schemaHints => 'RegionName STRING'
) limit 5;

In [0]:
(spark.readStream.format("cloudFiles")
   .option("cloudFiles.format", "csv")
   .option("header", "true")
   .option("cloudFiles.schemaLocation", "/Volumes/housing_dev/landing/raw/_schemas/zori")
   .option("cloudFiles.schemaHints", "RegionName STRING")
   .load("/Volumes/housing_dev/landing/raw/zori/")
 .writeStream
   .option("checkpointLocation", "/Volumes/housing_dev/landing/raw/_checkpoints/zori")
   .trigger(availableNow=True)
   .toTable("housing_dev.bronze.zori_raw"))

In [0]:
%sql
select count(*) from housing_dev.bronze.zori_raw;
describe history housing_dev.bronze.zori_raw;

In [0]:
%sql
SELECT * FROM housing_dev.bronze.zori_raw LIMIT 5;

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("housing_dev.bronze.zori_raw")
month_cols = [c for c in bronze.columns if c.startswith("20")]
id_cols= ["RegionName","City","Metro","CountyName","State"]

long_df = bronze.unpivot(

 ids=id_cols,
 values=month_cols,
 variableColumnName="month",
 valueColumnName="rent"   
)

display(long_df.limit(10))

In [0]:
clean_df = (long_df
    .filter(F.col("State") == "TX")                              # keep Texas only (Austin + Houston)
    .withColumn("month", F.to_date("month"))                     # text "2015-01-31" -> real date
    .withColumn("rent", F.round(F.col("rent").cast("double"), 2))# text -> number, 2 decimals
    .filter(F.col("rent").isNotNull())                           # drop months with no data
    .withColumnRenamed("RegionName", "zip")
    .withColumnRenamed("CountyName", "county")
    .select("zip", "City", "county", "Metro", "month", "rent")
    .dropDuplicates(["zip", "month"])                            # guarantee one row per ZIP per month
)

clean_df.write.mode("overwrite").saveAsTable("housing_dev.silver.zori_tx")

display(spark.table("housing_dev.silver.zori_tx").orderBy("zip", "month").limit(10))

In [0]:
%sql
select Metro, count(distinct zip) as zips, avg(rent) as rents, min(month) as First_Month, max(month) as Last_Month
from housing_dev.silver.zori_tx
group by metro 
order by zips desc


In [0]:
from pyspark.sql.window import Window

silver = spark.table("housing_dev.silver.zori_tx")


w= Window.partitionBy("zip").orderBy("month")

trend_df= (silver
    .withColumn("rent_last_month", F.lag("rent",1).over(w))
    .withColumn("rent_last_year", F.lag("rent",12).over(w))
    .withColumn("mom_pct", F.round((F.col("rent") / F.col("rent_last_month") - 1) * 100, 2))
    .withColumn("yoy_pct", F.round((F.col("rent") / F.col("rent_last_year") - 1) * 100, 2))

)

trend_df.write.mode("overwrite").saveAsTable("housing_dev.gold.zori_tx_trend_rent")

display(trend_df.filter(F.col("zip") == "78660").orderBy("month"))


In [0]:
%sql
SELECT zip, city, month,rent,
lag(rent,1) OVER (PARTITION BY zip ORDER BY month) as rent_last_year
from housing_dev.silver.zori_tx
where zip = '78660'
order by month DESC
limit 13;

In [0]:
%sql
select zip, city, month,rent, rent_last_year, yoy_pct
from housing_dev.gold.zori_tx_trend_rent
where Metro like 'Austin%'
and month = (select max(month) from housing_dev.gold.zori_tx_trend_rent)
and yoy_pct is not null
order by yoy_pct desc
limit 10;
  

In [0]:
%sql

CREATE OR REPLACE VIEW housing_dev.gold.v_austin_rent_drops AS
SELECT zip, City, month, rent, rent_last_year, yoy_pct
FROM housing_dev.gold.zori_tx_trend_rent
WHERE Metro LIKE 'Austin%'
  AND month = (SELECT MAX(month) FROM housing_dev.gold.zori_tx_trend_rent)
  AND yoy_pct IS NOT NULL;

SELECT * FROM housing_dev.gold.v_austin_rent_drops ORDER BY yoy_pct LIMIT 10;

In [0]:
from pyspark.sql import functions as F

# ---- BRONZE: Auto Loader (same as ZORI, different folder/table) ----
(spark.readStream.format("cloudFiles")
   .option("cloudFiles.format", "csv")
   .option("header", "true")
   .option("cloudFiles.schemaLocation", "/Volumes/housing_dev/landing/raw/_schemas/zhvi")
   .option("cloudFiles.schemaHints", "RegionName STRING")
   .load("/Volumes/housing_dev/landing/raw/zhvi/")
 .writeStream
   .option("checkpointLocation", "/Volumes/housing_dev/landing/raw/_checkpoints/zhvi")
   .trigger(availableNow=True)
   .toTable("housing_dev.bronze.zhvi_raw")
   .awaitTermination())        

# ---- SILVER: unpivot + clean (same as ZORI, value column is home_value) ----
bronze_v = spark.table("housing_dev.bronze.zhvi_raw")
month_cols_v = [c for c in bronze_v.columns if c.startswith("20")]

zhvi_silver = (bronze_v
    .unpivot(ids=["RegionName", "City", "Metro", "CountyName", "State"],
             values=month_cols_v,
             variableColumnName="month",
             valueColumnName="home_value")
    .filter(F.col("State") == "TX")
    .withColumn("month", F.to_date("month"))
    .withColumn("home_value", F.round(F.col("home_value").cast("double"), 0))
    .filter(F.col("home_value").isNotNull())
    .withColumnRenamed("RegionName", "zip")
    .select("zip", "month", "home_value")     
    .dropDuplicates(["zip", "month"]))

zhvi_silver.write.mode("overwrite").saveAsTable("housing_dev.silver.zhvi_tx")

display(spark.table("housing_dev.silver.zhvi_tx").filter(F.col("zip") == "78660").orderBy(F.desc("month")).limit(5))

In [0]:
from pyspark.sql import functions as F

rent = spark.table("housing_dev.silver.zori_tx")
value = spark.table("housing_dev.silver.zhvi_tx")

inner_df = rent.join(value, on=["zip", "month"], how="inner")
left_df = rent.join(value, on=["zip", "month"], how="left")

print("rent rows: ", rent.count())
print("inner join rows: ", inner_df.count())
print("left join rows: ", left_df.count())

print("rent rows with No Home values", left_df.filter(F.col("home_value").isNull()).count())


# DBTITLE 1)

In [0]:
rent_value = (inner_df
    .withColumn("price_to_rent", F.round(F.col("home_value") / (F.col("rent") * 12), 1))
    .withColumn("gross_yield_pct", F.round(F.col("rent") * 12 / F.col("home_value") * 100, 2)))

rent_value.write.mode("overwrite").option("mergeSchema", "true").saveAsTable("housing_dev.gold.rent_value_zip")


display(rent_value.
        filter(F.col("Metro").startswith("Austin")).
        filter(F.col("month") == rent_value.agg(F.max("month")).first()[0])
        .orderBy(F.desc("price_to_rent")))



In [0]:
metro_dim = spark.createDataFrame(
    [("Austin-Round Rock-Georgetown, TX",      "Central TX"),
     ("San Antonio-New Braunfels, TX",         "Central TX"),
     ("Houston-The Woodlands-Sugar Land, TX",  "Gulf Coast"),
     ("Dallas-Fort Worth-Arlington, TX",       "North TX"),
     ("El Paso, TX",                           "West TX")],
    ["Metro", "region"])

rent = spark.table("housing_dev.silver.zori_tx")

with_region = rent.join(F.broadcast(metro_dim), on="Metro", how="left")

with_region.explain()     # prints the physical plan

display(with_region.groupBy("region").agg(F.count_distinct("Metro").alias("metros"),
                                          F.count("*").alias("rows")))

In [0]:
rent = spark.table("housing_dev.silver.zori_tx")

a = rent.select("zip", "month", "rent").limit(3)
b = rent.select("month", "zip", "rent").limit(3)     

print("1) union – matches columns by POSITION")
display(a.union(b))

In [0]:
print("2) unionByName – matches columns by NAME")
display(a.unionByName(b))

In [0]:
c = rent.select("zip", "month").limit(2)            

print("3) unionByName with allowMissingColumns")
display(a.unionByName(c, allowMissingColumns=True))

In [0]:
print("4) Does union remove duplicates?")
print(a.union(a).count())              # 6 → NO, keeps duplicates
print(a.union(a).distinct().count())   # 3 → add .distinct() to dedupe

In [0]:
%sql
create or replace materialized view housing_dev.gold.mv_metro_monthly_rent as
SELECT Metro,
       month,
       ROUND(AVG(rent), 2)  AS avg_rent,
       COUNT(DISTINCT zip)  AS zips
       FROM housing_dev.silver.zori_tx
       GROUP BY Metro, month;

In [0]:
%sql
SELECT * FROM housing_dev.gold.mv_metro_monthly_rent
WHERE Metro LIKE 'Austin%'
ORDER BY month DESC LIMIT 12;


In [0]:
rent = spark.table("housing_dev.silver.zori_tx")

display(rent.groupBy("Metro").agg(
    F.count("*").alias("rows"),
    F.count_distinct("zip").alias("zips_exact"),
    F.approx_count_distinct("zip").alias("zips_approx"),
    F.round(F.avg("rent"), 0).alias("avg_rent"),
).orderBy(F.desc("rows")))

In [0]:
%sql
CREATE OR REFRESH STREAMING TABLE housing_dev.bronze.zori_st
AS SELECT *, _metadata.file_name AS source_file
FROM STREAM read_files(
  '/Volumes/housing_dev/landing/raw/zori/',
  format => 'csv',
  header => true,
  schemaHints => 'RegionName STRING'
);